In [23]:
# Try it out
# Load dataset

import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd

file_path = "./Titanic-Dataset.csv"

titanic_full: pd.DataFrame = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "yasserh/titanic-dataset",
  file_path,
)

print(titanic_full.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
None


In [24]:
# Shuffle and split a dataset
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(
    titanic_full, test_size=0.2, stratify=titanic_full["Pclass"], random_state=42
)
# X train
titanic = train_set.drop("Survived", axis=1)
# y train
titanic_survived_labels = train_set["Survived"].copy()

# X_test
test = test_set.drop("Survived", axis=1)
# y_test 
test_survived_labels = test_set["Survived"].copy()

In [25]:
# Transformation Pipelines
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

titanic_num = titanic.select_dtypes(include=[np.number])

# для числовых — медиана. SimpleImputer заполнение пропусков (age)
median_num_pipeline = make_pipeline(SimpleImputer(strategy="median"))

# для категориальных — самое частое значение (Embarked)
# а также делаем преобразование текста в цифры через OneHotEncoder так как более двух категорий
cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")
)

# Format text data to numerical data (encoding)
ordinal_enc_pipeline = make_pipeline(OrdinalEncoder())


# логаримф значения
# RuntimeWarning: divide by zero encountered in log ; result = func(self.values, **kwargs)
# Тут мы используем log1p потому что есть данные с нулями
# Их можно наполнить так как я делал в domain knowledge, но на этот раз проще делаем
log_pipeline = make_pipeline(
    FunctionTransformer(np.log1p, inverse_func=np.expm1, feature_names_out="one-to-one")
)

preprocessing = ColumnTransformer(
    [
        # Если не будет значения, то заполнить медианой, либо берем ориг
        ("median_num", median_num_pipeline, ["Pclass", "Age", "SibSp", "Parch"]),
        # Для Embarked заполняем тот что чаще встречается
        ("mf_num", cat_pipeline, ["Embarked"]),
        # Преобразуем в 0 и 1
        ("ord_enc", ordinal_enc_pipeline, ["Sex"]),
        # Логаримф для Fare (цены билета)
        ("log", log_pipeline, ["Fare"]),
    ]
)

In [32]:
# Для справки
column_prepared = preprocessing.fit_transform(titanic)
print(column_prepared[:2])

# Recover DataFrame
df_prepared = pd.DataFrame(
    column_prepared,
    columns=preprocessing.get_feature_names_out(),
    index=titanic.index,
)
print("features", df_prepared.shape)
print(df_prepared[:5])


[[ 1.         52.          1.          1.          0.          0.
   1.          0.          4.54859983]
 [ 2.         31.          0.          0.          0.          0.
   1.          1.          2.44234704]]
features (712, 9)
     median_num__Pclass  median_num__Age  median_num__SibSp  \
820                 1.0             52.0                1.0   
439                 2.0             31.0                0.0   
821                 3.0             27.0                0.0   
403                 3.0             28.0                1.0   
343                 2.0             25.0                0.0   

     median_num__Parch  mf_num__Embarked_C  mf_num__Embarked_Q  \
820                1.0                 0.0                 0.0   
439                0.0                 0.0                 0.0   
821                0.0                 0.0                 0.0   
403                0.0                 0.0                 0.0   
343                0.0                 0.0                 0.0

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

# Обучение модели

# Создаём pipeline:
# 1) preprocessing — превращает сырые данные (titanic) в числовые фичи
# 2) LogisticRegression — обучает модель на этих фичах
# STOP: TOTAL NO. of ITERATIONS REACHED LIMIT. получаем эту ошибку по этому делаем max_iter=1000
log_reg = make_pipeline(preprocessing, LogisticRegression(max_iter=1000))

# Обучение модели:
# titanic — входные данные
# titanic_survived_labels — реальные данные выживаемости (что нужно предсказать)
# Модель учится находить зависимость: признаки → выживаемость
log_reg.fit(titanic, titanic_survived_labels)

# Делаем предсказания на тех же данных (train set)
titanic_predictions = log_reg.predict(titanic)

# Выводим первые 5 предсказаний
print(
    "predict -\n",
    titanic_predictions[:30]
)

# Выводим реальные значения 
print(
    "real - \n",
    titanic_survived_labels.iloc[:30].values
)

predict -
 [1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 1 0 1 1 1 0 0 0 0 0 1 0]
real - 
 [1 0 1 0 0 0 0 0 0 1 0 1 0 0 0 1 0 0 1 1 1 1 0 0 0 0 0 0 1 0]


In [15]:
# Теперь замерим на сколько точно, но на этот раз с этим пайплайном
from sklearn.model_selection import cross_val_score

scores = cross_val_score(log_reg, titanic, titanic_survived_labels,
    scoring="accuracy", cv=10)

print(pd.Series(scores).describe())
# Результаты почти те же самый что и без пайплайна, только в этот
# раз у нас он более чистый и верный, так как Fare мы нормализовали
# учитываем Embarked так как привели его к OneHotEncoder

count    10.000000
mean      0.804871
std       0.052591
min       0.708333
25%       0.774648
50%       0.823944
75%       0.832746
max       0.859155
dtype: float64


Теперь попробуем разные модели

DecisionTreeRegressor можно было бы использовать , но он не подходит для 0 и 1
поэтому мы будем использовать DecisionTreeClassifier

In [27]:
# Try DecisionTreeClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

tree_class = make_pipeline(preprocessing, DecisionTreeClassifier(random_state=42))
tree_class.fit(titanic, titanic_survived_labels)
titanic_predictions = tree_class.predict(titanic)
scores = cross_val_score(tree_class, titanic, titanic_survived_labels,
    scoring="accuracy", cv=10)

print(pd.Series(scores).describe())
# DecisionTreeClassifier показал mean 0.795 против 0.804 
# у LogisticRegression. Чуть хуже.

count    10.000000
mean      0.794992
std       0.046176
min       0.732394
25%       0.764085
50%       0.802817
75%       0.816901
max       0.887324
dtype: float64


In [ ]:
# Try a RandomForestClassifier
from sklearn.ensemble import RandomForestClassifier

forest_class = make_pipeline(preprocessing, RandomForestClassifier(random_state=42))
scores = cross_val_score(forest_class, titanic, titanic_survived_labels,
    scoring="accuracy", cv=10)

pd.Series(scores).describe()
# RandomForestClassifier показал mean 0.816 — лучше всех предыдущих!

count    10.000000
mean      0.816060
std       0.047349
min       0.732394
25%       0.784038
50%       0.825215
75%       0.845070
max       0.873239
dtype: float64

In [ ]:
# Try a SVC (Support Vector Classifier)
# 
# Это просто другой алгоритм классификации — альтернатива RandomForest и LogisticRegression.
# Идея: найти линию (или кривую) которая лучше всего разделяет выживших и погибших.
#
# Зачем пробовать разные модели?
# Нет универсально лучшего алгоритма — на разных данных побеждают разные модели.
# Поэтому пробуем несколько и выбираем победителя по accuracy.
#
# Результат без подбора параметров: mean 65% — хуже чем RandomForest (84%)
# После GridSearchCV с kernel="rbf", C=10: 80% — лучше, но всё равно проигрывает
#
# Вывод: для Titanic RandomForest оказался лучшим

from sklearn.svm import SVC

svc = make_pipeline(preprocessing, SVC(random_state=42))
scores = cross_val_score(svc, titanic, titanic_survived_labels,
    scoring="accuracy", cv=10)

pd.Series(scores).describe()

count    10.000000
mean      0.654499
std       0.034441
min       0.605634
25%       0.627201
50%       0.647887
75%       0.687745
max       0.704225
dtype: float64

In [38]:
# Grid Search
# поиск оптимального гиперпараметра
# searches for the best combination of hyperparameter values for the RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

full_pipeline = Pipeline(
    [
        ("preprocessing", preprocessing),
        ("random_forest", RandomForestClassifier(random_state=42)),
    ]
)
# Сами параметры которые передаются в GridSearchCV
param_grid = {
	# Имя референс по full_pipeline смотри!
	# Пробуем разное кол-во деревьев в лесу
    "random_forest__n_estimators": [50, 100, 200],
	# Пробуем разную глубинку
    "random_forest__max_depth": [None, 5, 10],
}

# 9 * 10 = 90 раз попробует, будет делить трейн сет и проверять
# измеряет ошибку по accuracy и смотрит лучшее
grid_search = GridSearchCV(
    full_pipeline, param_grid, cv=10, scoring="accuracy"
)

grid_search.fit(titanic, titanic_survived_labels)
print(grid_search.best_params_)
print(grid_search.best_score_)
# Лучший результат 0.8370892018779343 !

# evaluation scores
cv_res = pd.DataFrame(grid_search.cv_results_)
cv_res.sort_values(by="mean_test_score", ascending=False, inplace=True)
cv_res.head() # note: the 1st column is the row ID

{'random_forest__max_depth': 10, 'random_forest__n_estimators': 100}
0.8370892018779343


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_random_forest__max_depth,param_random_forest__n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,split5_test_score,split6_test_score,split7_test_score,split8_test_score,split9_test_score,mean_test_score,std_test_score,rank_test_score
7,0.091485,0.001083,0.006578,0.000156,10,100,"{'random_forest__max_depth': 10, 'random_fores...",0.833333,0.833333,0.830986,0.873239,0.887324,0.915493,0.830986,0.760563,0.845070,0.760563,0.837089,0.046751,1
8,0.179336,0.001484,0.009279,0.000181,10,200,"{'random_forest__max_depth': 10, 'random_fores...",0.833333,0.847222,0.845070,0.873239,0.859155,0.915493,0.859155,0.760563,0.816901,0.760563,0.837070,0.045528,2
6,0.048997,0.000521,0.005116,0.000120,10,50,"{'random_forest__max_depth': 10, 'random_fores...",0.861111,0.833333,0.816901,0.873239,0.901408,0.873239,0.830986,0.774648,0.845070,0.746479,0.835642,0.044653,3
3,0.046582,0.000712,0.005246,0.000195,5,50,"{'random_forest__max_depth': 5, 'random_forest...",0.847222,0.805556,0.859155,0.845070,0.845070,0.873239,0.873239,0.746479,0.774648,0.802817,0.827250,0.040793,4
0,0.056336,0.011280,0.006527,0.003055,None,50,"{'random_forest__max_depth': None, 'random_for...",0.861111,0.805556,0.830986,0.845070,0.873239,0.887324,0.859155,0.774648,0.802817,0.718310,0.825822,0.048890,5


In [47]:
# Try SVC with GridSearchCV
from sklearn.svm import SVC

svc_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("svc", SVC(random_state=42)),
])

param_grid = {
    "svc__kernel": ["linear", "rbf"],
    "svc__C": [0.1, 1, 10],
    # gamma только для rbf
    "svc__gamma": ["scale", "auto"],
}

grid_search_svc = GridSearchCV(svc_pipeline, param_grid, cv=3, scoring="accuracy")
grid_search_svc.fit(titanic, titanic_survived_labels)
print(grid_search_svc.best_params_)
print(grid_search_svc.best_score_)

{'svc__C': 10, 'svc__gamma': 'scale', 'svc__kernel': 'rbf'}
0.8034074389249372


In [ ]:
# Randomized Search
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_distribs = {
    "random_forest__n_estimators": randint(low=50, high=300),
    "random_forest__max_depth": randint(low=2, high=30),
}

rnd_search = RandomizedSearchCV(
    full_pipeline,
    param_distributions=param_distribs,
    n_iter=10,
    cv=10,
    scoring="accuracy",
    random_state=42,
)
rnd_search.fit(titanic, titanic_survived_labels)
# Analyzing the Best Models and Their Errors
final_model = rnd_search.best_estimator_  # includes preprocessing
feature_importances = final_model["random_forest"].feature_importances_
print("feature_importances\n", feature_importances.round(2))
# Sort importance scores
print(
    "scores\n",
    sorted(
        zip(feature_importances, final_model["preprocessing"].get_feature_names_out()),
        reverse=True,
    ),
)
# Топ по важности:
# Sex — 0.35 (самый важный)
# Fare — 0.22
# Age — 0.19
# Pclass — 0.10

# Так как был fit мы можем узнать по rnd_search лучшей рез
print(rnd_search.best_score_)
# 0.8497456964006259 лучше чем GridSearchCV!

feature_importances
 [0.1  0.19 0.06 0.04 0.01 0.01 0.02 0.35 0.22]
scores
 [(0.3511758868408542, 'ord_enc__Sex'), (0.21934948014563038, 'log__Fare'), (0.18977320400612352, 'median_num__Age'), (0.09922346064894855, 'median_num__Pclass'), (0.059965562952477645, 'median_num__SibSp'), (0.04164025005257462, 'median_num__Parch'), (0.015905150424759856, 'mf_num__Embarked_S'), (0.013428782098091993, 'mf_num__Embarked_C'), (0.009538222830539159, 'mf_num__Embarked_Q')]
0.8497456964006259


In [45]:
# Evaluate Your System on the Test Set
from sklearn.metrics import accuracy_score
from scipy import stats

x_test = test
y_test = test_survived_labels

final_predictions = final_model.predict(x_test)
final_score = accuracy_score(y_test, final_predictions)
print(final_score) # 0.8212290502793296

# train cross-validation score — 84.9%. 
# Разница небольшая, это хороший знак — модель не переобучилась.

# confidence interval используем доверительный интервал чтобы убедиться
# на сколько все хорошо

# correct.reduce((sum, val) => sum + val, 0) / correct.length
def accuracy(predictions_and_labels):
    # predictions_and_labels это массив где 1 = угадал, 0 = не угадал
    return np.mean(predictions_and_labels)

confidence = 0.95
# промапить массив и оставить 1 или 0 если сходится, условно 
# (pred, i) => pred === yTest[i] ? 1 : 0
correct = (final_predictions == y_test.values).astype(int)

# много раз пересэмплирует, каждый раза вызывает accuracy(sample)
# получает разные accuracy
boot_result = stats.bootstrap(
    [correct], accuracy, confidence_level=confidence, random_state=42
)
rmse_lower, rmse_upper = boot_result.confidence_interval
# С процентом 95% реальная ошибка может быть такая (лучше, хуже)
print("rmse_lower-", rmse_lower)
print("rmse_upper-", rmse_upper)


0.8212290502793296
rmse_lower- 0.7597765363128491
rmse_upper- 0.8715083798882681
